In [0]:
dbutils.widgets.dropdown("data_source", "customers", ["customers", "orders"], "Data Source")
dbutils.widgets.dropdown("catalog", "dev", ["dev", "prod"])

data_source = dbutils.widgets.get("data_source")
catalog = dbutils.widgets.get("catalog")

print(f"Selected source: {data_source} and catalog: {catalog}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DateType, TimestampType, DoubleType

from pyspark.sql.window import Window

from delta.tables import DeltaTable

In [0]:
SOURCE_TABLE_MAP = {
    "customers": "bronze_customers_valid",
    "orders": "bronze_orders_valid",
}

TARGET_TABLE_MAP = {
    "customers": "silver_customers_scd2",
    "orders": "silver_orders_scd2",
}

source_table = f"{catalog}.os_stepright.{SOURCE_TABLE_MAP[data_source]}"
target_table = f"{catalog}.os_stepright.{TARGET_TABLE_MAP[data_source]}"

In [0]:
print(f"source_table is-> [{source_table}] And target_table is-> [{target_table}]")

In [0]:
silver_customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("gender", StringType(), True),
    StructField("registration_date", TimestampType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("address_line1", StringType(), True),
    StructField("address_line2", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True),
    StructField("valid_from", TimestampType(), True),
    StructField("valid_to", TimestampType(), True),
])

In [0]:
silver_orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_date", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("shipping_address_id", StringType(), True),
    StructField("shipping_city", StringType(), True),
    StructField("shipping_state", StringType(), True),
    StructField("shipping_country", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("discount_code", StringType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True),
    StructField("valid_from", TimestampType(), True),
    StructField("valid_to", TimestampType(), True),
])

In [0]:
def target_table_existence_check(target_table, data_source):

    if data_source=='customers':
        silver_table_schema= silver_customers_schema
    else:
        silver_table_schema = silver_orders_schema

    if not spark.catalog.tableExists(target_table):

        empty_df = spark.createDataFrame([], silver_table_schema)
        empty_df.write.format("delta").saveAsTable(target_table)
    
    else:
        pass

In [0]:
customer_tracked_cols = ["email", "first_name", "last_name", "phone", "date_of_birth",
                "gender", "registration_date", "loyalty_tier", "address_line1",
                "address_line2", "city", "state", "zip_code", "country", "is_active"]

In [0]:
orders_tracked_cols = ["customer_id", "order_status", "order_date", "updated_at",
                 "shipping_address_id", "shipping_city", "shipping_state",
                 "shipping_country", "payment_method", "discount_code",
                 "discount_amount", "total_amount"]

In [0]:
if data_source=='customers':
    tracked_cols= customer_tracked_cols
else:
    tracked_cols= orders_tracked_cols

In [0]:
changed_condition = " OR ".join([f"src.{c} <> tgt.{c}" for c in tracked_cols])
print(changed_condition)

In [0]:
no_changed_condition = " AND ".join([f"src.{col} = tgt.{col}" for col in tracked_cols])
print(no_changed_condition)

#### 1) Load source_table incoming source_df

In [0]:
def read_source_table(source_table):

    source_df = (spark.table(source_table)
             .selectExpr('after.*', 'op', 'ts_ms', '_source_file', '_ingested_at')
             .withColumn("ts_ms", (F.col("ts_ms") / 1000).cast("timestamp"))
    )

    return source_df

#### 2) deduplicate incoming source_df

In [0]:
def dedup_incoming_source_df(source_df, data_source):

    if data_source=='customers':
        partition_on_pk= "customer_id"
    else:
        partition_on_pk= "order_id"

    window_spec = Window.partitionBy(F.col(partition_on_pk)).orderBy(F.col("ts_ms").desc())

    dedup_source_df = (source_df.withColumn("rn", F.row_number().over(window_spec))
                   .filter("rn=1")
                   .drop("rn")
    )

    return dedup_source_df

#### 3) splitting incoming source_df into buckets like deletes_df,changed_rows,new_rows,no_op_changes

In [0]:
def dedup_incoming_source_split_into_buckets(dedup_source_df, target_table,data_source, changed_condition,no_changed_condition):

    if data_source=='customers':
        src_pk = "src.customer_id"
        tgt_pk = "tgt.customer_id"
    else:
        src_pk = "src.order_id"
        tgt_pk = "tgt.order_id"
    

    tgt_df = spark.table(target_table).filter("valid_to='9999-12-31'")

    deletes_df = dedup_source_df.where("op='d'")

    dedup_source_op_cu_df= dedup_source_df.filter("op in ('c', 'u')")

    changed_rows = (dedup_source_op_cu_df.alias("src")
                .join(tgt_df.alias("tgt"), F.col(src_pk) == F.col(tgt_pk), "inner")
                .where(changed_condition)
    ).selectExpr("src.*")

    new_rows = (dedup_source_op_cu_df.alias("src")
            .join(tgt_df.alias("tgt"), F.col(src_pk) == F.col(tgt_pk), "left_anti")
                
    ).selectExpr("src.*")

    no_op_changes = (dedup_source_op_cu_df.alias("src")
                 .join(tgt_df.alias("tgt"), F.col(src_pk) == F.col(tgt_pk), "inner")
                .where(no_changed_condition)
    ).selectExpr("src.*")

    return deletes_df, changed_rows, new_rows, no_op_changes


#### 4) writing to target

In [0]:
def write_to_target(data_source, target_table,deletes_df, changed_rows, new_rows):

    if data_source=='customers':
        merge_cond = "t.customer_id = s.customer_id AND t.valid_to = '9999-12-31'"

    else:
        merge_cond = "t.order_id = s.order_id AND t.valid_to = '9999-12-31'"

    
    if data_source=='customers':
        pk= "customer_id"
    else:
        pk= "order_id"


    if data_source=='customers':
        s_pk = "s.customer_id"
        t_pk = "t.customer_id"
    else:
        s_pk = "s.order_id"
        t_pk = "t.order_id"

 # merged records getting merged   

    target_delta = DeltaTable.forName(spark, target_table)

    retire_source= deletes_df.unionByName(changed_rows)

    (target_delta.alias("t")
    .merge(retire_source.alias("s"), merge_cond)
    .whenMatchedUpdate(set= {
     "valid_to": "s.ts_ms"
    })
    .execute()
    )

# insert records getting written
    rows_to_insert = changed_rows.unionByName(new_rows)

    rows_to_insert_final = (
    rows_to_insert
    .withColumn("valid_from", F.col("ts_ms"))
    .withColumn("valid_to", F.lit("9999-12-31").cast("timestamp"))
    .withColumn("valid_from", F.col("valid_from").cast("timestamp"))
    .drop("op", "ts_ms")
)

    existing_target = spark.table(target_table).select(pk, "valid_from")

    final_insert_df = (rows_to_insert_final.alias("s")
    .join(
        existing_target.alias("t"),
        (F.col(s_pk) == F.col(t_pk)) &
        (F.col("s.valid_from") == F.col("t.valid_from")),
        "left_anti"
    )
)

    final_insert_df.write.format("delta").mode("append").saveAsTable(target_table)

#### 5) calling all functions

In [0]:
target_table_existence_check(target_table, data_source)

source_df= read_source_table(source_table)

dedup_source_df = dedup_incoming_source_df(source_df, data_source)

deletes_df, changed_rows, new_rows, no_op_changes = dedup_incoming_source_split_into_buckets(dedup_source_df, target_table,data_source, changed_condition,no_changed_condition)

write_to_target(data_source, target_table,deletes_df, changed_rows, new_rows)



In [0]:
spark.sql(f"select * from {target_table}").display()

In [0]:
spark.sql(f"drop table {target_table}"")